# 02 · Hypothesis Testing

Submit hypotheses to the Veritas API and run statistical comparisons against the corpus.

> **[MOCK]** Hypothesis payloads below use placeholder field names.  
> Update `HYPOTHESIS_PAYLOAD` to match real field names after the physicist session.

In [ ]:
import os
import json
import requests
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
API_BASE = os.getenv('VERITAS_API_BASE', 'http://localhost:5000')
CORPUS_ID = os.getenv('VERITAS_CORPUS_ID', 'demo-lenr-corpus')  # [MOCK]

In [ ]:
# [MOCK] Replace with a real hypothesis payload matching the LENR ontology
HYPOTHESIS_PAYLOAD = {
    'statement': 'Higher loading ratios (>0.90) correlate with measurable excess heat',
    'parameters': {
        'loading_ratio': {'operator': 'gt', 'value': 0.90}  # [MOCK] operator values TBD with physicist
    },
    'domain_pack_id': 'lenr-v1'  # [MOCK] update when pack is registered
}

resp = requests.post(
    f'{API_BASE}/api/corpora/{CORPUS_ID}/hypothesis/test',
    json=HYPOTHESIS_PAYLOAD
)
if resp.ok:
    hypothesis_result = resp.json()
    print(json.dumps(hypothesis_result, indent=2))
else:
    # [MOCK] Fallback when API is not running
    hypothesis_result = {
        'hypothesis_id': 'mock-hyp-001',
        'statement': HYPOTHESIS_PAYLOAD['statement'],
        'support_count': 7,
        'contradict_count': 2,
        'inconclusive_count': 3,
        'confidence_score': 0.65,
        'verdict': 'supported'
    }
    print('[MOCK] Using fallback data')

In [ ]:
# Visualise support/contradict/inconclusive breakdown
labels = ['Support', 'Contradict', 'Inconclusive']
counts = [
    hypothesis_result.get('support_count', 0),
    hypothesis_result.get('contradict_count', 0),
    hypothesis_result.get('inconclusive_count', 0),
]
colours = ['#4ade80', '#f87171', '#facc15']

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, counts, color=colours)
ax.set_ylabel('Document count')
ax.set_title(f"Hypothesis: {hypothesis_result.get('statement', '')[:60]}…")
plt.tight_layout()
plt.savefig('hypothesis_result.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [MOCK] Chi-square goodness-of-fit: expected even distribution across categories
# Replace with domain-informed expected ratios after physicist session
total = sum(counts)
if total > 0:
    expected = [total / 3, total / 3, total / 3]  # [MOCK] placeholder expected distribution
    chi2, p = stats.chisquare(f_obs=counts, f_exp=expected)
    print(f'χ² = {chi2:.3f}, p = {p:.4f}')
    if p < 0.05:
        print('Significant deviation from uniform distribution (p < 0.05)')
    else:
        print('No significant deviation from uniform distribution')